## Notebook for testing our pipeline for the Generative Recommender

In [1]:
import sys
print(sys.executable)

/opt/anaconda3/envs/gen-rec/bin/python


In [2]:
# Imports here
from DataHandler import get_article_feature_string_list
from embedder import Embedder
from rqvae import RQVAE
from rqvae_train import rqvae_loss, train_rqvae_sanity_check, train_rqvae_full, load_trained_rqvae
import torch
import numpy as np
import pickle
import os
import joblib
import torch, numpy as np, random
import pandas as pd
from transformer import get_all_unique_sids, prepare_dataset, train_model, recommended_next_sid, is_model_trained
from transformers import BartTokenizer, BartForConditionalGeneration
from collaborative_filtering import compute_item_user_matrix, recommend_for_user
from evaluation import MAP_at_K

# This is ONLY for our tests! Do not use for our full model
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

### First, get the feature strings from DataHandler

In [3]:
feature_strings = get_article_feature_string_list()
# feature_strings is a list of lists, use list comprehension to make it a list of strings (first 1000 items)
feature_strings = [item[0] for item in feature_strings]
item_ids = [feature_string.split()[0] for feature_string in feature_strings]

print(f'Number of items: {len(feature_strings)}')
print(feature_strings[0])


Number of items: 105542
108775015 Solid Dark Black Strap top Jersey top with narrow shoulder straps. Vest top Garment Upper body Jersey Basic Ladieswear Ladieswear Womens Everyday Basics Jersey Basic


### Next, we use the embedder to get the embeddings that will serve as input for RQ-VAE

In [4]:
if os.path.exists('SBERT_embeddings_fulldata.npy'):
    embeddings = np.load('SBERT_embeddings_fulldata.npy')
    print(f'The embeddings have the shape: {embeddings.shape}')
else:
    embedder = Embedder("sbert")
    embeddings = embedder.encode(feature_strings) # embeddings has shape (n, d), where d = 384 for SBERT
    print(f'The embeddings have the shape: {embeddings.shape}')
    # print(embeddings[0])
    np.save('SBERT_embeddings_fulldata.npy', embeddings)

The embeddings have the shape: (105542, 384)


### Now, we can use the embeddings with RQ-VAE. The code loads hashmaps between item IDs and semantic IDs if they exist. If not, it uses the model to generate semantic IDs and creates (and saves) the hashmaps

In [5]:
input_dim = embeddings.shape[1]
latent_dim = 32
hidden_dim = 256
codebook_size = 512
num_quantizers = 4

TRAINING_MODE = 'None' # for loading

set_seed(42)
rqvae = RQVAE(input_dim=input_dim, latent_dim=latent_dim, hidden_dim=hidden_dim, codebook_size=512, num_quantizers=num_quantizers)

# We need to train the model here, before retrieving semantic IDs! Omitting for now... 
# trained_model = ...

print('Generating semantic IDs')

if isinstance(embeddings, np.ndarray):
    embeddings = torch.from_numpy(embeddings).float()

if TRAINING_MODE == 'sanity_check':
    print("Running sanity check for RQ-VAE")
    rqvae = train_rqvae_sanity_check(rqvae, embeddings)

elif TRAINING_MODE == 'full_training':
    print("Training RQ-VAE")
    rqvae = train_rqvae_full(rqvae, embeddings, save_path='./models/rqvae')

else:
    rqvae = load_trained_rqvae(rqvae, 'quantizers/rqvae_training_results_20251101/models/rqvae_20251101_best.pth')

semantic_ids = rqvae.encode_to_semantic_ids(embeddings)
semantic_ids = semantic_ids.detach().numpy() # List of semantic IDs

print(f'semantic_ids has shape: {semantic_ids.shape}')
print('printing the first semantic ID (trained RQ-VAE)')
print(semantic_ids[0])

item_2_semantic = {}
semantic_2_item = {}

if os.path.exists('quantizers/rqvae_training_results_20251101/item_2_semantic.pkl') and os.path.exists('quantizers/rqvae_training_results_20251101/semantic_2_item.pkl'):
    print('Loading existing hashmaps')

    with open('quantizers/rqvae_training_results_20251101/item_2_semantic.pkl', 'rb') as f:
        item_2_semantic = pickle.load(f)
    with open('quantizers/rqvae_training_results_20251101/semantic_2_item.pkl', 'rb') as f:
        semantic_2_item = pickle.load(f)
    print('Loaded the hashmaps')

else:
    for item_id, semantic_id in zip(item_ids, semantic_ids):
        semantic_tuple = tuple(semantic_id.astype(int))
        item_2_semantic[item_id] = semantic_tuple
        semantic_2_item[semantic_tuple] = item_id

    print(f'Done creating hashmaps. Saving...')
    with open('item_2_semantic.pkl', 'wb') as f:
        pickle.dump(item_2_semantic, f)

    with open('semantic_2_item.pkl', 'wb') as f:
        pickle.dump(semantic_2_item, f)
    print(f'Saved hashmaps to files.')

Generating semantic IDs
semantic_ids has shape: (105542, 4)
printing the first semantic ID (trained RQ-VAE)
[201  91 394  18]
Loading existing hashmaps
Loaded the hashmaps


### Converting transactions data into: 
#### i.e.: user_id: item_id1, ..., item_idn --> user_id: item_SID1, ..., item_SIDn

In [6]:
transactions_train = pd.read_pickle('transaction_list_train.pkl')
transactions_val = pd.read_pickle('transaction_list_val.pkl')

customer_transactions_train = {}
customer_transactions_val = {}
if os.path.exists('customer_transactions_train.pkl') and os.path.exists('customer_transactions_val.pkl'):
    print('Loading existing customer_transactions_train and customer_transactions_val')

    with open('customer_transactions_train.pkl', 'rb') as f:
        customer_transactions_train = pickle.load(f)
    with open('customer_transactions_val.pkl', 'rb') as f:
        customer_transactions_val = pickle.load(f)

    print('Loaded existing customer_transactions_train and customer_transactions_val')
else:

    for customer_id, group in transactions_train.groupby("customer_id"):
        item_ids = group["article_id"].tolist()[0]
        sid_list = []
        for item_id in item_ids:
            sid = item_2_semantic.get(str(item_id))
            if sid is not None:
                sid_list.append(sid)
            else:
                print(f"Warning: missing mapping for article_id {item_id}.")
        customer_transactions_train[customer_id] = sid_list
    
    for customer_id, group in transactions_val.groupby("customer_id"):
        item_ids = group["article_id"].tolist()[0]
        sid_list = []
        for item_id in item_ids:
            sid = item_2_semantic.get(str(item_id))
            if sid is not None:
                sid_list.append(sid)
            else:
                print(f"Warning: missing mapping for article_id {item_id}.")
        customer_transactions_val[customer_id] = sid_list
    
    print("customer_transactions_train and customer_transactions_val have been created, saving...")
    with open("customer_transactions_train.pkl", "wb") as f:
        pickle.dump(customer_transactions_train, f)
    with open('customer_transactions_val.pkl', 'wb') as f:
        pickle.dump(customer_transactions_val, f)
    print("Saving done!")

first_customer = list(customer_transactions_val.keys())[0]
print(f"{first_customer}: {customer_transactions_val[first_customer]}")

Loading existing customer_transactions_train and customer_transactions_val
Loaded existing customer_transactions_train and customer_transactions_val
00000dbacae5abe5e23885899a1fa44253a17956c6d1c3d25f88aa139fdfc657: [(np.int64(139), np.int64(283), np.int64(20), np.int64(354)), (np.int64(462), np.int64(79), np.int64(375), np.int64(252)), (np.int64(288), np.int64(108), np.int64(30), np.int64(348)), (np.int64(218), np.int64(202), np.int64(120), np.int64(403)), (np.int64(502), np.int64(423), np.int64(113), np.int64(70)), (np.int64(502), np.int64(423), np.int64(113), np.int64(70)), (np.int64(321), np.int64(423), np.int64(330), np.int64(94)), (np.int64(381), np.int64(122), np.int64(356), np.int64(231)), (np.int64(260), np.int64(261), np.int64(303), np.int64(53)), (np.int64(502), np.int64(219), np.int64(312), np.int64(310)), (np.int64(502), np.int64(219), np.int64(312), np.int64(310)), (np.int64(303), np.int64(219), np.int64(120), np.int64(30)), (np.int64(318), np.int64(249), np.int64(505), np

### Taking a subset of transaction data (customer_transations_train, customer_transactions_val) for training of the transfomer

In [7]:
train_subset_keys = list(customer_transactions_train.keys())[:200]
customer_transactions_train_subset = {k: customer_transactions_train[k] for k in train_subset_keys}
print("First sample of train data: ", customer_transactions_train_subset[list(customer_transactions_train_subset.keys())[0]])

val_subset_keys = list(customer_transactions_val.keys())[:50]
customer_transactions_val_subset = {k: customer_transactions_val[k] for k in val_subset_keys}
print("First sample of val data: ", customer_transactions_val_subset[list(customer_transactions_val_subset.keys())[0]])

First sample of train data:  [(np.int64(139), np.int64(283), np.int64(20), np.int64(354)), (np.int64(462), np.int64(79), np.int64(375), np.int64(252)), (np.int64(288), np.int64(108), np.int64(30), np.int64(348)), (np.int64(218), np.int64(202), np.int64(120), np.int64(403)), (np.int64(502), np.int64(423), np.int64(113), np.int64(70)), (np.int64(502), np.int64(423), np.int64(113), np.int64(70)), (np.int64(321), np.int64(423), np.int64(330), np.int64(94)), (np.int64(381), np.int64(122), np.int64(356), np.int64(231)), (np.int64(260), np.int64(261), np.int64(303), np.int64(53)), (np.int64(502), np.int64(219), np.int64(312), np.int64(310)), (np.int64(502), np.int64(219), np.int64(312), np.int64(310)), (np.int64(303), np.int64(219), np.int64(120), np.int64(30)), (np.int64(318), np.int64(249), np.int64(505), np.int64(343)), (np.int64(289), np.int64(41), np.int64(357), np.int64(258)), (np.int64(482), np.int64(272), np.int64(112), np.int64(24)), (np.int64(63), np.int64(436), np.int64(492), np.in

### Training the transformer

In [8]:

if is_model_trained(): 
    print('The model is loaded...')
    model = BartForConditionalGeneration.from_pretrained('./bart-recommender/final_model')
    tokenizer = BartTokenizer.from_pretrained('./bart-recommender/final_model')
else: 
    print('There is no pretrained model, the model will be trained ...')

    tokenizer = BartTokenizer.from_pretrained('facebook/bart-base')
     # ensure pad token exists (single-token)
    if tokenizer.pad_token is None:
        tokenizer.add_special_tokens({"pad_token": "<PAD_ITEM>"})

    tokenizer.padding_side = "left"

    # collect all SIDs and add them (single tokens)
    sids_train = get_all_unique_sids(customer_transactions_train_subset)
    sids_val = get_all_unique_sids(customer_transactions_val_subset)
    all_sids = list(set(sids_train) | set(sids_val))
    # add tokens
    tokenizer.add_tokens(all_sids)

    model = BartForConditionalGeneration.from_pretrained("facebook/bart-base")
    # resize embeddings after we changed tokenizer
    model.resize_token_embeddings(len(tokenizer))
    model.config.pad_token_id = tokenizer.pad_token_id

    train_dataset = prepare_dataset(customer_transactions_train_subset, window_size=35, tokenizer=tokenizer)
    val_dataset = prepare_dataset(customer_transactions_val_subset, window_size=36, tokenizer=tokenizer)
    train_model(train_dataset, model, val_dataset)

    # save
    os.makedirs("./bart-recommender/final_model", exist_ok=True)
    model.save_pretrained('./bart-recommender/final_model')
    tokenizer.save_pretrained('./bart-recommender/final_model')

The model is loaded...


### Retrieving articles info for mapping

In [9]:
articles = pd.read_pickle('articles.pkl')
article_id = 108775044
row = articles.loc[articles['article_id'] == article_id]
product_name = row['prod_name']
product_name2 = row['prod_name'].iloc[0]
print(product_name)
print(product_name2)

1    Strap top
Name: prod_name, dtype: object
Strap top


### Inference: 

### GR: recommend SIDs for given customer transaction sequence

In [13]:
last_10_customers = list(customer_transactions_val.keys())[-20:]
test_customers_sequences = {k: customer_transactions_val[k] for k in last_10_customers}

for customer_id, seqs in test_customers_sequences.items():
    test_seq = test_customers_sequences[customer_id]
    test_seq = [' '.join(tuple(str(x) for x in sid)) for sid in test_seq]
    print(f"Customer: {customer_id}")
    print(f"Input SIDs sequence: {test_seq}")
    recommended_sids = recommended_next_sid(test_seq, model, tokenizer, window_size=36, top_k=12)
    filtered = [ # filter out empty sids
        (sid, semantic_2_item[tuple(int(x) for x in sid.split())])
        for sid in recommended_sids
        if sid.strip() != ''
    ]

    print("Recommended SIDs, generated by transformer: ")
    for sid, id in filtered:
        row = articles.loc[articles['article_id'].astype(str) == str(id)]
        prod_name = row['prod_name'].iloc[0]
        print(f'Rec. SID: {sid} --> article_id: {id} --> product_name: {prod_name}')

Customer: fffd82f00fc748fae98fa569c6863a4c5b2242e0c8162871a91bcbf10b4b95be
Input SIDs sequence: ['380 17 47 365', '404 352 335 108', '212 204 382 510', '145 352 223 70', '202 368 120 305', '202 368 120 305', '202 368 120 305']
Recommended SIDs, generated by transformer: 
Rec. SID: 343 151 233 311 --> article_id: 706016001 --> product_name: Jade HW Skinny Denim TRS
Rec. SID: 448 247 291 256 --> article_id: 706016050 --> product_name: Jade HW Skinny Denim TRS
Rec. SID: 469 279 350 193 --> article_id: 372860001 --> product_name: 7p Basic Shaftless
Rec. SID: 479 80 31 29 --> article_id: 464297033 --> product_name: Greta Thong Mynta Low 3p
Rec. SID: 333 196 244 404 --> article_id: 759871002 --> product_name: Tilda tank
Rec. SID: 469 279 90 133 --> article_id: 372860002 --> product_name: 7p Basic Shaftless
Rec. SID: 83 398 120 457 --> article_id: 610776002 --> product_name: Tilly (1)
Rec. SID: 169 366 97 499 --> article_id: 688537004 --> product_name: Simple as that Cheeky Tanga
Rec. SID: 44

### 2. Collaborative Filtering:

In [11]:
if os.path.exists('CF_cashe.joblib'):
    unique_items, user2idx, interaction_matrix, item_similarity = joblib.load('CF_cashe.joblib')
else:
    unique_items, user2idx, interaction_matrix, item_similarity = compute_item_user_matrix()
    joblib.dump((unique_items,user2idx, interaction_matrix, item_similarity), "CF_cashe.joblib")

for customer_id in test_customers_sequences:
    recommended_items = recommend_for_user(customer_id, unique_items, user2idx, interaction_matrix, item_similarity, top_k=12)
    print(f"Customer: {customer_id}")
    for article_id in recommended_items:
        row = articles.loc[articles['article_id'].astype(str) == str(article_id)]
        prod_name = row['prod_name'].iloc[0]
        print(f"article_id: {id} --> product_name: {prod_name}")


Customer: ffff11185bb6dc52d546424a28657dcb3c891b1f57e27b84d6538a08c57aaee1
article_id: 741356009 --> product_name: Fidde tee(1)
article_id: 741356009 --> product_name: Goldie tank top
article_id: 741356009 --> product_name: Fidde tee(1)
article_id: 741356009 --> product_name: Goldie tank top
article_id: 741356009 --> product_name: Fidde tee
article_id: 741356009 --> product_name: Bird ss
article_id: 741356009 --> product_name: Goldie tank top
article_id: 741356009 --> product_name: Fidde tee
article_id: 741356009 --> product_name: Fidde tee(1)
article_id: 741356009 --> product_name: Seb overshirt
article_id: 741356009 --> product_name: Goldie tank top
article_id: 741356009 --> product_name: Tilda tank
Customer: ffff12aa623c69eae8959d673f1f12ad0194ad760d77fd489cd7c5a4aa9ae240
article_id: 741356009 --> product_name: Daiquiri Pull- On TRS
article_id: 741356009 --> product_name: Nira tee
article_id: 741356009 --> product_name: Skinny Ankle R.W Brooklyn
article_id: 741356009 --> product_nam

# Evaluation Code
1. Extract all data of length 8 (Done)
2. transform data to cust_id->sem_id_list dictionary (Done)
3. Tranfrom data to cust_id-> (input, target) (Done
4. Load transformer (Done)
5. Run prediction for Generative recommendation
6. Run prediction for Collaborative filtering

In [12]:
# Recommend K products to U customers
# Save all predictions in a U x K np.array
pred = None

# Have a U x 1 np.array with labels
lab = None

# Evaluate using both pred and lab
map12 = MAP_at_K(pred, lab)

TypeError: 'NoneType' object is not iterable

1. Item-> semantic

In [ ]:
# Imports here
import DataHandler as dh
from embedder import Embedder
from rqvae import RQVAE
from rqvae_train import rqvae_loss, train_rqvae_sanity_check, train_rqvae_full, load_trained_rqvae
import torch
import numpy as np
import pickle
import os
import joblib
import torch, numpy as np, random
import pandas as pd
from transformer import get_all_unique_sids, prepare_dataset, train_model, recommended_next_sid, is_model_trained
from transformers import BartTokenizer, BartForConditionalGeneration
from collaborative_filtering import compute_item_user_matrix, recommend_for_user
from evaluation import MAP_at_K

#bencmark libs
import time
# Part 0 Loading semantic id hashmaps
item_2_semantic = {}
semantic_2_item = {}

if os.path.exists('item_2_semantic.pkl') and os.path.exists('semantic_2_item.pkl'):
    print('Loading existing hashmaps')

    with open('item_2_semantic.pkl', 'rb') as f:
        item_2_semantic = pickle.load(f)
    with open('semantic_2_item.pkl', 'rb') as f:
        semantic_2_item = pickle.load(f)
    print('Loaded the hashmaps')

# Extract all transactions with length 8
transactions_train = pd.read_pickle('customer_transactions_train60P.pkl')
print(len(transactions_train))
transactions_eval = dh.user_profile_extract_equal_to_threshold(transactions_train, 8)

#########################################################################################################
# Part 1 Convert or load  cust_id -> sem_id_list

customer_transactions_eval = {}
if os.path.exists('customer_transactions_eval.pkl'):
    print('Loading existing customer_transactions_eval and customer_transactions_val')

    with open('customer_transactions_eval.pkl', 'rb') as f:
        customer_transactions_eval = pickle.load(f)

    print('Loaded existing customer_transactions_eval and customer_transactions_val')
else:

    for customer_id, group in transactions_eval.groupby("customer_id"):
        item_ids = group["article_id"].tolist()[0]
        sid_list = []
        for item_id in item_ids:
            sid = item_2_semantic.get(str(item_id))
            if sid is not None:
                sid_list.append(sid)
            else:
                print(f"Warning: missing mapping for article_id {item_id}.")
        customer_transactions_eval[customer_id] = sid_list

    print("customer_transactions_eval has been created, saving...")
    with open("customer_transactions_eval.pkl", "wb") as f:
        pickle.dump(customer_transactions_eval, f)
    print("Saving done!")

# Load and set parameters of model
window_size = 3

if is_model_trained():
    print('The model is loaded...')
    model = BartForConditionalGeneration.from_pretrained('./bart-recommender/final_model')
    tokenizer = BartTokenizer.from_pretrained('./bart-recommender/final_model')

# Create dictioniuoary cust_id -> (input, target)
transaction_eval_target_input = dh.split_input_label_transactions(customer_transactions_eval, input_size=3, labels=5)
sampled_10_percent = random.sample(list(transaction_eval_target_input.items()),
                              int(0.01 * len(transaction_eval_target_input)))
precision_sum = 0
recall_sum = 0
elapsed_time_sum = 0
counter = 0;
for customer_id, (input, target) in sampled_10_percent:
    start = time.time()
    input = [' '.join(tuple(str(x) for x in sid)) for sid in input]
    target = [' '.join(tuple(str(x) for x in sid)) for sid in target]
    #print(f"Customer: {customer_id}")
    #print(f"INPUT sequence: {input}")
    #print(f"Target sequences: {target}")
    start = time.time()
    recommended_sids = recommended_next_sid(input, model, tokenizer, window_size, top_k=5)
    print(f"recomendish:{time.time() - start}")
    #print(f"Prediction: {recommended_sids}")
    #filtered = [  # filter out empty sids
    #    (sid, semantic_2_item[tuple(int(x) for x in sid.split())])
    #    for sid in recommended_sids
    #    if sid.strip() != ''
    #]
    precision_at_k = evaluation.precision_at_K(recommended_sids, target, 5)
    precision_sum += precision_at_k
    recall_at_k = evaluation.recall_at_K(recommended_sids, target, 5)
    recall_sum += recall_at_k
    if(counter % 10 == 0):
        print(f"# iterations {counter}") 
    counter = counter + 1
    elapsed_time_sum += time.time() - start 
    '''
    print("Recommended SIDs, generated by transformer: ")
    for sid, id in filtered:
        row = articles.loc[articles['article_id'].astype(str) == str(id)]
        prod_name = row['prod_name'].iloc[0]
        print(f'Rec. SID: {sid} --> article_id: {id} --> product_name: {prod_name}')
    '''
print(f"Mean time for an iteration at 5: {elapsed_time_sum / len(sampled_10_percent)}")
print(f"Mean Precisision at 5: {precision_sum / len(sampled_10_percent)}")
print(f"Mean recall at 5: {recall_sum / len(sampled_10_percent)}")


Loading existing hashmaps
Loaded the hashmaps
738460
Loading existing customer_transactions_eval and customer_transactions_val
Loaded existing customer_transactions_eval and customer_transactions_val
The model is loaded...
recomendish:2.28611159324646
# iterations 0
recomendish:0.41603779792785645
recomendish:0.3859670162200928
recomendish:0.4219322204589844
recomendish:0.38172197341918945
recomendish:0.39078831672668457
recomendish:0.40308356285095215
recomendish:0.4022939205169678
recomendish:0.39295315742492676
recomendish:0.37293481826782227
recomendish:0.39639782905578613
# iterations 10
recomendish:0.38890552520751953
recomendish:0.3336169719696045
recomendish:0.36526060104370117
recomendish:0.34760141372680664
recomendish:0.3567013740539551
recomendish:0.3459494113922119
recomendish:0.3405308723449707
recomendish:0.42674779891967773
recomendish:0.6105389595031738
recomendish:0.3742947578430176
# iterations 20
recomendish:0.4317786693572998
recomendish:0.37975168228149414
recomen

## 2. Full collaborative Filtering Evaluation code

In [ ]:

precision_sum = 0
recall_sum = 0
elapsed_time_sum = 0
if os.path.exists('CF_cashe.joblib'):
    unique_items, user2idx, interaction_matrix, item_similarity = joblib.load('CF_cashe.joblib')
else:
    unique_items, user2idx, interaction_matrix, item_similarity = compute_item_user_matrix()
    joblib.dump((unique_items,user2idx, interaction_matrix, item_similarity), "CF_cashe.joblib")

for customer_id, (input, target) in sampled_10_percent:
    start = time.time()

    recommended_items = recommend_for_user(customer_id, unique_items, user2idx, interaction_matrix, item_similarity, top_k=5)
    #print(f"prediction{recommended_items}")
    #print(f"target{target_items}")    
    prediction_items = [str(sid) for sid in recommended_items]
    target_items = [semantic_2_item[semid] for semid in target]
    

    precision_at_k = evaluation.precision_at_K(prediction_items, target_items, 5)
    precision_sum += precision_at_k
    recall_at_k = evaluation.recall_at_K(prediction_items, target_items, 5)
    recall_sum += recall_at_k
    elapsed_time_sum += time.time() - start 

    #print(f"Customer: {customer_id}")
    #for article_id in recommended_items:
        #row = articles.loc[articles['article_id'].astype(str) == str(article_id)]
        #prod_name = row['prod_name'].iloc[0]
        #print(f"article_id: {id} --> product_name: {prod_name}")
print(f"Mean time for an iteration at 5: {elapsed_time_sum / len(sampled_10_percent)}")

print(f"Mean Precisision at 5: {precision_sum / len(sampled_10_percent)}")
print(f"Mean recall at 5: {recall_sum / len(sampled_10_percent)}")

Mean time for an iteration at 5: 0.006688747295113497
Mean Precisision at 5: 0.003875968992248062
Mean recall at 5: 0.003875968992248062
